In [1]:
import pandas as pd
import numpy as np
import re

# loading the dataset
df = pd.read_excel('UseCase - Airlines.xlsx')
df.shape

(1020, 7)

In [2]:
df.head()

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


In [3]:
df.dtypes

flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

In [4]:
# checking null values
df.isnull().sum()

flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

In [5]:
# checking duplicate rows
df.duplicated().sum()

np.int64(15)

In [6]:
df['airline'].unique()
#checking values in the airline column

array(['SpiceJet', 'Air India', 'Vistara', 'IndiGo', 'UNKNOWN', nan],
      dtype=object)

In [7]:
# checking if source/dest codes are consistent
df['source'].unique(), df['destination'].unique()

(array(['CCU', 'BOM', 'MAA', 'DEL', 'BLR', 'HYD'], dtype=object),
 array(['MAA', 'CCU', 'BOM', 'HYD', 'BLR', 'DEL'], dtype=object))

In [8]:
# cleaning up whitespace and casing
df['flight_id'] = df['flight_id'].astype(str).str.strip().str.upper()
df['airline'] = df['airline'].astype(str).str.strip()
df['source'] = df['source'].astype(str).str.strip().str.upper()
df['destination'] = df['destination'].astype(str).str.strip().str.upper()

# treating invalid and missing values in airline column
df['airline'] = df['airline'].replace(['UNKNOWN', 'nan', ''], np.nan)
df['airline'].value_counts(dropna=False)

airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           72
Name: count, dtype: int64

In [9]:
# checking flight_id format
pat = re.compile(r'^[A-Z0-9]{2}\d{3}$')
valid_id = df['flight_id'].apply(lambda x: bool(pat.match(x)))
valid_id.value_counts()

flight_id
True    1020
Name: count, dtype: int64

In [10]:
df[~valid_id]

,flight_id,airline,source,destination,departure_time,arrival_time,duration


In [11]:
# a lot of airline values are missing
df['prefix'] = df['flight_id'].str[:2]
df['prefix'].value_counts()
#but flight_id prefix tells us the airline anyway so we will be using it to fix that

prefix
6F    273
AI    260
SJ    251
UK    236
Name: count, dtype: int64

In [12]:
# mapping prefix to airline name to fill the gaps
prefix_map = {'AI':'Air India', 'SJ':'SpiceJet', 'UK':'Vistara', '6F':'IndiGo'}

missing = df['airline'].isna()
df.loc[missing, 'airline'] = df.loc[missing, 'prefix'].map(prefix_map)

df['airline'].isna().sum()

np.int64(0)

In [13]:
# checking for datetime objects in depttime and arrivaltime columns
df['departure_time'] = pd.to_datetime(df['departure_time'], errors='coerce')
df['arrival_time'] = pd.to_datetime(df['arrival_time'], errors='coerce')

df[df['departure_time'].isna() | df['arrival_time'].isna()]

,flight_id,airline,source,destination,departure_time,arrival_time,duration,prefix


In [14]:
# arrival date earlier than departure date is garbage data
bad_time = df['arrival_time'] < df['departure_time']
bad_time.sum()

np.int64(1)

In [15]:
df[bad_time]

,flight_id,airline,source,destination,departure_time,arrival_time,duration,prefix
355,SJ192,SpiceJet,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,SJ


In [16]:
# dropping corrupted row
print(len(df))
df = df[~bad_time].reset_index(drop=True)
print(len(df))

1020
1019


In [17]:
# the actual overnight case
df['overnight'] = df['arrival_time'].dt.date != df['departure_time'].dt.date
df['overnight'].sum()

np.int64(124)

In [18]:
# since arrival_time already carries the correct date even for overnight flights we use same logic overall
df['duration_min'] = (df['arrival_time'] - df['departure_time']).dt.total_seconds() / 60
df[['departure_time','arrival_time','overnight','duration_min']].sample(10)

,departure_time,arrival_time,overnight,duration_min
275,2026-04-20 02:03:41.703,2026-04-20 04:25:41.703,False,142.0
742,2026-04-18 12:07:41.703,2026-04-18 13:43:41.703,False,96.0
299,2026-04-19 23:35:41.702,2026-04-20 00:22:41.702,True,47.0
212,2026-04-20 07:00:41.701,2026-04-20 08:49:41.701,False,109.0
496,2026-04-19 07:53:41.701,2026-04-19 08:35:41.701,False,42.0
826,2026-04-18 05:09:41.703,2026-04-18 09:38:41.703,False,269.0
512,2026-04-19 06:10:41.703,2026-04-19 10:52:41.703,False,282.0
511,2026-04-19 06:16:41.702,2026-04-19 10:24:41.702,False,248.0
630,2026-04-18 21:51:41.703,2026-04-18 22:57:41.703,False,66.0
530,2026-04-19 05:14:41.701,2026-04-19 09:59:41.701,False,285.0


In [19]:
# get rid of duplicates
print(len(df))
df = df.drop_duplicates(subset=['flight_id','departure_time','arrival_time'])
print(len(df))

1019
1004


In [20]:
# flag weird durations - zero/negative shouldn't exist anymore but checking anyway, anything over 6hrs is sus for domestic routes
df['anomaly'] = (df['duration_min'] <= 0) | (df['duration_min'] > 360)
df['anomaly'].sum()

np.int64(0)

In [21]:
df[df['anomaly']]

,flight_id,airline,source,destination,departure_time,arrival_time,duration,prefix,overnight,duration_min,anomaly


In [22]:
df['route'] = df['source'] + '-' + df['destination']
df['dep_date'] = df['departure_time'].dt.date
df['dep_hour'] = df['departure_time'].dt.hour

df_final = df[['flight_id','airline','source','destination','route',
               'departure_time','arrival_time','dep_date','dep_hour',
               'overnight','duration_min','anomaly']]
df_final.head()

,flight_id,airline,source,destination,route,departure_time,arrival_time,dep_date,dep_hour,overnight,duration_min,anomaly
0,SJ010,SpiceJet,CCU,MAA,CCU-MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,2026-04-20,23,True,174.0,False
1,AI155,Air India,BOM,CCU,BOM-CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,2026-04-20,23,True,108.0,False
2,UK094,Vistara,BOM,CCU,BOM-CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,2026-04-20,23,True,105.0,False
3,AI245,Air India,BOM,CCU,BOM-CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,2026-04-20,23,True,156.0,False
4,AI192,Air India,MAA,BOM,MAA-BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,2026-04-20,23,True,299.0,False


In [23]:
df_final['duration_min'].mean()

np.float64(164.48519332669326)

In [24]:
df_final.groupby('airline')['duration_min'].mean().sort_values(ascending=False)

airline
Air India    165.388352
IndiGo       164.945182
Vistara      164.278433
SpiceJet     163.231828
Name: duration_min, dtype: float64

In [25]:
df_final['route'].value_counts().head(10)

route
BOM-CCU    90
CCU-DEL    72
MAA-BLR    65
BLR-BOM    60
HYD-MAA    57
DEL-HYD    54
HYD-DEL    42
BOM-DEL    39
CCU-BOM    33
DEL-BLR    29
Name: count, dtype: int64

In [26]:
df_final.groupby('airline')['anomaly'].sum()

airline
Air India    0
IndiGo       0
SpiceJet     0
Vistara      0
Name: anomaly, dtype: int64

In [27]:
df_final['dep_hour'].value_counts().sort_index()

dep_hour
0     27
1     35
2     42
3     41
4     35
5     35
6     31
7     32
8     34
9     29
10    45
11    40
12    37
13    58
14    59
15    54
16    59
17    43
18    39
19    34
20    55
21    47
22    49
23    44
Name: count, dtype: int64

In [28]:
# formatting datetime columns cleanly before export so power bi reads them correctly
df_final['departure_time'] = df_final['departure_time'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_final['arrival_time'] = df_final['arrival_time'].dt.strftime('%Y-%m-%d %H:%M:%S')

df_final.to_csv('flights_cleaned.csv', index=False)

C:\Users\rstar\AppData\Local\Temp\ipykernel_33488\2599114560.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['departure_time'] = df_final['departure_time'].dt.strftime('%Y-%m-%d %H:%M:%S')
C:\Users\rstar\AppData\Local\Temp\ipykernel_33488\2599114560.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['arrival_time'] = df_final['arrival_time'].dt.strftime('%Y-%m-%d %H:%M:%S')


In [29]:
df_final[['departure_time', 'arrival_time']].head()

,departure_time,arrival_time
0,2026-04-20 23:38:41,2026-04-21 02:32:41
1,2026-04-20 23:35:41,2026-04-21 01:23:41
2,2026-04-20 23:26:41,2026-04-21 01:11:41
3,2026-04-20 23:07:41,2026-04-21 01:43:41
4,2026-04-20 23:05:41,2026-04-21 04:04:41
